[Reference](v)

# The Core Philosophy

## 1. Signatures define your task’s input/output contract:
```
# Simple: "question -> answer"
# With types: "question -> answer: float"
# Complex: "context, question -> response"
```

## 2. Modules describe the strategy for invoking your LM:

In [1]:
# Basic prediction
predict = dspy.Predict("question -> answer")

# With reasoning
cot = dspy.ChainOfThought("question -> answer")
# Agent with tools
agent = dspy.ReAct("question -> answer", tools=[search, calculate])

## 3. Optimizers automatically tune prompts and weights:

In [2]:
optimizer = dspy.MIPROv2(metric=your_metric, auto="light")
optimized_program = optimizer.compile(your_module, trainset=data)

# A Concrete Example

In [3]:
import dspy
# Configure your LM
lm = dspy.LM("openai/gpt-4o-mini")
dspy.configure(lm=lm)
# Define the module (two lines!)
math = dspy.ChainOfThought("question -> answer: float")
# Use it
result = math(question="Two dice are tossed. What is the probability that the sum equals two?")
print(result.reasoning)
print(result.answer)  # 0.0277776

In [4]:
optimizer = dspy.MIPROv2(metric=exact_match, auto="light")
optimized_math = optimizer.compile(math, trainset=trainset)

# Building a RAG Pipeline

In [5]:
class RAG(dspy.Module):
    def __init__(self, num_docs=5):
        self.num_docs = num_docs
        self.respond = dspy.ChainOfThought("context, question -> response")

    def forward(self, question):
        context = search(question, k=self.num_docs)
        return self.respond(context=context, question=question)

# Optimize with semantic F1
optimizer = dspy.MIPROv2(
    metric=dspy.evaluate.SemanticF1(decompositional=True),
    auto="medium"
)
optimized_rag = optimizer.compile(RAG(), trainset=trainset)

# Agent Development

In [6]:
def search_wikipedia(query: str) -> list[str]:
    results = dspy.ColBERTv2(url="http://...")(query, k=3)
    return [x["text"] for x in results]

def evaluate_math(expression: str):
    return dspy.PythonInterpreter({}).execute(expression)
react = dspy.ReAct(
    "question -> answer: float",
    tools=[search_wikipedia, evaluate_math]
)
result = react(question="What is 9362158 divided by the year David Gregory was born?")

# When to Use DSPy

In [7]:
pip install -U dspy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 312.4/312.4 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 69.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 17.5 MB/s eta 0:00:00


In [8]:
import dspy
lm = dspy.LM("openai/gpt-4o-mini", api_key="...")
dspy.configure(lm=lm)